# Aula 5 — Pandas II: limpeza, transformação e padronização de dados
**Fund. de Programação, Dados e Estatística para IA** 

**Ideia central:** entender o problema, definir uma regra, transformar e conferir o resultado. Preserve sempre a base original.

### Preparação: criando a base da aula
A base fictícia é criada a partir de um dicionário e salva como `pacientes_aula5_bruto.csv`. Os problemas são **intencionais**: linha repetida, espaços e maiúsculas em `servico`, idade escrita como texto, valores ausentes, datas em formatos diferentes e readmissão com várias representações.

In [ ]:
import pandas as pd

dados_brutos = {
    "id_paciente":            [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 102],
    "idade":                  ["68", " 45 ", "72", "cinquenta", "59", "150", "33", None, "81", "52", " 45 "],
    "tempo_internacao_dias":  ["5", "3", "7 dias", "4", "6", "2", "1 dia", "8", "10", "3", "3"],
    "internacoes_anteriores": ["2", "0", "3", "1", "não informado", "0", "1", "dois", "4", "-", "0"],
    "servico":                ["Cardiologia", "Clínica Médica", "cardiologia", "Neurologia", " CARDIOLOGIA ",
                               "clinica medica", "Ortopedia", "Neurologia", "Cardiologia ", "Clínica Médica", "Clínica Médica"],
    "readmissao_30d":         ["Sim", "Não", "sim", "0", "1", "Não", "nao", "1", "Sim", "0", "Não"],
    "data_alta":              ["15/03/2026", "18/03/2026", "2026-03-20", "22/03/2026", "25/03/2026", "27-03-2026",
                               "31/02/2026", "05/04/2026", "-", "10/04/2026", "18/03/2026"]
}

pd.DataFrame(dados_brutos).to_csv("pacientes_aula5_bruto.csv", index=False)
print("Arquivo pacientes_aula5_bruto.csv criado.")

---
## 01 · Carregar e preservar
### `read_csv()` e `copy()`

In [ ]:
df_bruto = pd.read_csv("pacientes_aula5_bruto.csv")
df = df_bruto.copy()

df.head()

`df_bruto` é o dado como foi recebido; `df` é a cópia de trabalho, que recebe as alterações.

---
## 02 · Ausências e duplicidades
### `isna()` e `sum()`: contar ausências

In [ ]:
df.isna().sum()

> **Ausência não é zero.** Nesta aula só identificamos a ausência; não preenchemos com média ou zero.

### `duplicated()` e `drop_duplicates()`

In [ ]:
df[df.duplicated(keep=False)]   # mostra todas as ocorrências repetidas

In [ ]:
df = df.drop_duplicates()        # remove duplicatas completas
df.shape

> **Cuidado:** só remova depois de definir o que é duplicidade. Um paciente pode ter várias internações legítimas.

---
## 03 · Textos e categorias
### `str.strip()` e `str.lower()`: limpar texto

In [ ]:
df["servico_aux"] = df["servico"].str.strip().str.lower()
df[["servico", "servico_aux"]]

### `unique()` e `map()`: padronizar categorias

In [ ]:
df["servico_aux"].unique()

In [ ]:
mapa = {"cardiologia": "Cardiologia",
        "clinica medica": "Clínica Médica",
        "clínica médica": "Clínica Médica",
        "neurologia": "Neurologia"}

df["servico_padronizado"] = df["servico_aux"].map(mapa)
df[["servico", "servico_padronizado"]]

Valores fora do mapa ficam ausentes (veja *Ortopedia*). Vamos localizá-los no bloco 05.

---
## 04 · Converter tipos
### `to_numeric()`

In [ ]:
df["idade_num"] = pd.to_numeric(
    df["idade"], errors="coerce"
)
df[["idade", "idade_num"]]

### `to_datetime()`

In [ ]:
df["data_alta_dt"] = pd.to_datetime(
    df["data_alta"], dayfirst=True, errors="coerce"
)
df[["data_alta", "data_alta_dt"]]

`dayfirst=True` lê o dia antes do mês. Com `errors="coerce"`, o que não pode ser convertido vira ausente — inclusive datas em outro formato. Confira sempre o formato de origem.

---
## 05 · Conferir e salvar
### `notna()`, `isna()` e `loc`: verificar falhas
Uma falha é um valor que **existia no original** e **ficou ausente após a conversão**. `&` significa "e".

In [ ]:
falha = df["idade"].notna() & df["idade_num"].isna()
df.loc[falha, ["id_paciente", "idade"]]

O mesmo padrão serve para categorias fora do mapa:

In [ ]:
falha_servico = df["servico_aux"].notna() & df["servico_padronizado"].isna()
df.loc[falha_servico, ["id_paciente", "servico"]]

### `to_csv()`: salvar a base preparada

In [ ]:
df_limpo = df[["id_paciente", "idade_num",
               "servico_padronizado", "data_alta_dt"]].copy()

df_limpo.to_csv("pacientes_aula5_limpo.csv", index=False)
df_limpo.head()

> Uma conversão sem erro de execução ainda pode produzir valores que precisam de revisão.

---
## 06 · Prática: converter e conferir
1. Padronize `readmissao_30d` com `strip()`, `lower()` e `map()` para `True`/`False`.
2. Converta `internacoes_anteriores` e liste os valores que falharam.
3. Liste as linhas em que `data_alta` não foi convertida.

### Soluções